<center>

# ⠀
---

# **Analyse Numérique**

## **Méthodes itératives de résolution de systèmes linéaires**


---
<br>

**MANSOURI Koussay**



<br>


</center>

---

## I - Programmation et test de méthodes itératives

### I.1 - Méthode de Jacobi

> Créer la fonction `resol_syst_jac(A,b,prec,nitermax)` qui retourne le couple `(x,niter)`, où $x$ est la solution du système $Ax = b$ calculée grâce à la méthode de Jacobi, `niter` est le nombre d’itérations nécessaires pour atteindre la précision `prec`. La donnée `nitermax` est le nombre maximal d’itérations autorisé, au-delà duquel le programme s’arrête avec un message indiquant la non-convergence.
Tester votre fonction avec
> $$ A =
\begin{pmatrix}
2 & -1 & 0 \\
-1 & 2 & -1 \\
0 & -1 & 2
\end{pmatrix}
\quad \text{et} \quad b =
\begin{pmatrix}
1 \\
0 \\
1
\end{pmatrix}$$



Commençons par importer nos meilleurs amis pour la suite du TP !

In [ ]:
from math import *
import numpy as np
import matplotlib.pyplot as plt
import time

Afin d'appliquer la méthode, on décomposera la matrice $A$ comme suit :

$$A := D - E - F$$

Ici, $D$ est la matrice diagonale de $A$, $-E$ la matrice triangulaire inférieure de diagonale nulle tandis que $-F$ est la triangulaire supérieure.

Ici, on choisira $M = D$ et $N = E+F$ afin d'appliquer itérativement :

$$x^{(k+1)} = D^{-1}(E+F)x^k + D^{-1}b$$

In [ ]:
def resol_syst_jac(A,b,prec,nitermax):
    D = np.diag(np.diag(A))
    E = -np.tril(A, -1)
    F = -np.triu(A, 1)

    n = len(b)
    x = np.zeros(n)    # on le fixe arbitrairement
    niter = 0

    # inversion de D (sans np.linalg.inv(D) car ce serait tricher !)
    D_inv = np.zeros((n, n))
    for i in range(n):
        D_inv[i,i] = 1/D[i,i]

    # pour la précision, on vérif que ||Ax-b|| soit proche de 0
    while niter < nitermax and max(abs(A @ x - b)) > prec :
        x = (D_inv @ (((E + F) @ x) + b))
        niter += 1

    if niter == nitermax :
        print("Pas de convergence")

    return x, niter

In [ ]:
# TEST ---
A = np.array([[2, -1, 0], [-1, 2, -1], [0, -1, 2]])
b = np.array([1, 0, 1])
x, niter = resol_syst_jac(A, b, 1e-8, 100)

print("Matrice A :\n", A)
print("Second membre b :", b)
print(f"Solution x, trouvée en {niter} itérations :", x)

Matrice A :
 [[ 2 -1  0]
 [-1  2 -1]
 [ 0 -1  2]]
Second membre b : [1 0 1]
Solution x, trouvée en 54 itérations : [0.99999999 0.99999999 0.99999999]


---


### I.2 - Méthode de Gauss-Seidel

>Créer de même la fonction `resol_syst_gs(A,b,prec,nitermax)` qui utilise la méthode de Gauss-Seidel.
>
>Tester votre fonction avec les mêmes données que précédemment.

La méthode de Gauss-Seidel fonctionne de la même manière, mais il faut choisir $M = D - E$ et $N = F$.

In [ ]:
# On recycle le code tu TP2 !
def resol_syst_inf(A,b) :
  n = len(b)
  x = np.zeros(n)
  for i in range(n):
    somme = sum(A[i, j] * x[j] for j in range(i))
    x[i] = (b[i] - somme) / A[i, i]
  return x

In [ ]:
def resol_syst_gs(A,b,prec,nitermax):
    D = np.diag(np.diag(A))
    E = -np.tril(A, -1)
    F = -np.triu(A, 1)

    n = len(b)
    x = np.zeros(n)    # on le fixe arbitrairement
    niter = 0

    # pour la précision, on vérif que ||Ax-b|| soit proche de 0
    # et par précision, on travaille avec la norme inf
    while niter < nitermax and max(abs(A @ x - b)) > prec :
        x_temp = x.copy()
        x_temp = resol_syst_inf(D-E, (F @ x) + b)
        x = x_temp.copy()
        niter += 1

    if niter == nitermax :
        print("Pas de convergence")

    return x, niter

In [ ]:
# TEST ---
x, niter = resol_syst_gs(A, b, 1e-8, 100)

print("Matrice A :\n", A)
print("Second membre b :", b)
print(f"Solution x, trouvée en {niter} itérations :", x)

Matrice A :
 [[ 2 -1  0]
 [-1  2 -1]
 [ 0 -1  2]]
Second membre b : [1 0 1]
Solution x, trouvée en 28 itérations : [0.99999999 0.99999999 1.        ]
Matrice A :
 [[ 2 -1  0]
 [-1  2 -1]
 [ 0 -1  2]]
Second membre b : [1 0 1]
Solution x, trouvée en 28 itérations : [0.99999999 0.99999999 1.        ]


C'est bon !

---


### I.3 - Matrice tridiagonale

> Créer la fonction `mat_tridiag(n)` qui renvoie la matrice carrée $A_n$ de taille $n$ avec des $2$ sur la diagonale et des $-1$ sur les sur- et sous-diagonales.

In [ ]:
def mat_tridiag(n):
    M = np.zeros((n, n))
    np.fill_diagonal(M, 2)
    np.fill_diagonal(M[:-1, 1:], -1)
    np.fill_diagonal(M[1:, :-1], -1)
    return M

In [ ]:
# TEST ---
mat_tridiag(3)

array([[ 2., -1.,  0.],
       [-1.,  2., -1.],
       [ 0., -1.,  2.]])

---


### I.4 - Un vecteur quelconque

> Créer la fonction `vect(n)` qui renvoie le vecteur colonne $b_n$ de longueur $n$ défini par
> $$b_1 = 1, b_2 = b_3 = \dots = b_{n-1} = 0, b_n = 1$$

In [ ]:
def vect(n):
    M = np.zeros(n)
    M[0] = 1
    M[1:-1] = 0
    M[-1] = 1
    return M

In [ ]:
# TEST ---
vect(3)

array([1., 0., 1.])

---


### I.5 - Quelques résolutions de systèmes

> Pour différentes valeurs de $n$, résoudre le système $Ax = b$ où $A$ et $b$ sont respectivement définis aux questions 3 et 4 en utilisant les méthodes de Jacobi et de Gauss-Seidel. Commenter les résultats obtenus.

In [ ]:
# TEST ---
print("Avec la méthode de Jacobi :")
for n in range(4,21,4):
    A = mat_tridiag(n)
    b = vect(n)
    x, niter = resol_syst_jac(A, b, 1e-8, 10000)
    print(f"Pour n={n} : solution x trouvée en {niter} itérations.")

print("\nAvec la méthode de Gauss-Seidel :")
for n in range(4,21,4):
    A = mat_tridiag(n)
    b = vect(n)
    x, niter = resol_syst_gs(A, b, 1e-8, 10000)
    print(f"Pour n={n} : solution x trouvée en {niter} itérations.")

Avec la méthode de Jacobi :
Pour n=4 : solution x trouvée en 84 itérations.
Pour n=8 : solution x trouvée en 266 itérations.
Pour n=12 : solution x trouvée en 536 itérations.
Pour n=16 : solution x trouvée en 890 itérations.
Pour n=20 : solution x trouvée en 1323 itérations.

Avec la méthode de Gauss-Seidel :
Pour n=4 : solution x trouvée en 43 itérations.
Pour n=8 : solution x trouvée en 135 itérations.
Pour n=12 : solution x trouvée en 270 itérations.
Pour n=16 : solution x trouvée en 447 itérations.
Pour n=20 : solution x trouvée en 663 itérations.


Force est de constater que le nombre d'itérations de la méthode de Jacobi est bien supérieur que celui de la méthode de Gauss-Seidel, peu importe la valeur $n$. Cela reste toutefois un cas particulier, car on traite avec des matrices tridiagonales.

---

## II - Résolution par la méthode de relaxation et gradient à pas fixe
>
>On considère le vecteur$b\in \R^n$ et la matrice $\Lambda = (a_{ij})_{ij} \in \mathcal M_{n,n}(\R)$ définis à l'exercice 1. On admet que le spectre de la matrice $\Lambda$ de taille $n$ est donné par :
>$$Sp(\Lambda) = \bigcup_{i=1}^n \{\lambda_i\} = \{ \lambda_1, \dots, \lambda_n\}$$
>
>avec $\lambda_i = 2 - 2\cos(\frac{i\pi}{n+1}), 1 \leqslant i \leqslant n$

### II.1 - Méthode de relaxation

> On se donne $ω ∈ R^∗$. On rappelle que la méthode de relaxation est donnée par l’algorithme
suivant : $X(0)$ donné $\in \R^n$
>$$ (\frac{D}{\omega}-E) X^{(k+1)} = (\frac{1-\omega}{\omega}D + F)X^{(k)} + b, k\geqslant 0$$
>avec $E = (-a_{ij})_{i>j}, F = (-a_{ij})_{i<j}$ et $D = diag(a_{11}, \dots, a_{nn})$

---

#### II.1.1 - Un petite comparaison

> 1. **Pour $\omega = 1$**  
> Dans ce cas, la méthode de relaxation s'appelle également méthode de Gauss-Seidel.  
>   Justifier la convergence des méthodes de Jacobi et de Gauss-Seidel.  
>
>2. **Comparaison des vitesses de convergence**  
>   Laquelle des deux méthodes converge le plus vite ? Interpréter et comparer avec les résultats obtenus dans l’exercice 1.  
>
>**Paramètres à utiliser :**  
>- $n = 4, 8, 12, 16, 20$  
>
>**Indication :**  
>On note $J$ la matrice d’itération de Jacobi, et $G$ la matrice d’itération de Gauss-Seidel.  
>Dans le cas particulier où la matrice $A$ est tridiagonale, on admet que :  
>$$
>\rho(G) = (\rho(J))^2
>$$

In [ ]:
# Test pour différentes valeurs de n
n_values = [4, 8, 12, 16, 20]
prec = 1e-6
nitermax = 1000

print("Comparaison des méthodes Jacobi et Gauss-Seidel\n")
for n in n_values:
    A = mat_tridiag(n)
    b = vect(n)

    _, iter_jacobi = resol_syst_jac(A, b, prec, nitermax)
    _, iter_gs = resol_syst_gs(A, b, prec, nitermax)

    print(f"Pour n = {n} :")
    print(f"  Jacobi : {iter_jacobi} itérations")
    print(f"  Gauss-Seidel : {iter_gs} itérations")
    print("-" * 40)

Comparaison des méthodes Jacobi et Gauss-Seidel

Pour n = 4 :
  Jacobi : 62 itérations
  Gauss-Seidel : 33 itérations
----------------------------------------
Pour n = 8 :
  Jacobi : 192 itérations
  Gauss-Seidel : 98 itérations
----------------------------------------
Pour n = 12 :
  Jacobi : 380 itérations
  Gauss-Seidel : 192 itérations
----------------------------------------
Pour n = 16 :
  Jacobi : 622 itérations
  Gauss-Seidel : 313 itérations
----------------------------------------
Pour n = 20 :
  Jacobi : 913 itérations
  Gauss-Seidel : 458 itérations
----------------------------------------


---

#### II.1.2 - La  fonction de relaxation
>Créer une fonction `relaxation(A, b, prec, nitermax, $omega)` qui retourne un couple `(x, niter)`, où :  
>- $x$ est une approximation de la solution du système $Ax = b$ obtenue grâce à la méthode de relaxation, avec le paramètre $\omega$.
>- $niter$ est le nombre d’itérations nécessaires pour atteindre la précision `prec`.
>
>Si le nombre d'itérations dépasse `nitermax`, le programme doit s'arrêter avec un message indiquant la non-convergence.

In [ ]:
def relaxation(A, b, prec, nitermax, omega):
    n = len(b)
    x = np.zeros_like(b)  # Initialisation de la solution
    for k in range(nitermax):
        x_new = np.copy(x)
        for i in range(n):
            sigma1 = np.dot(A[i, :i], x_new[:i])
            sigma2 = np.dot(A[i, i+1:], x[i+1:])
            x_new[i] = (1 - omega) * x[i] + omega * (b[i] - sigma1 - sigma2) / A[i, i]

        # Vérification de la précision
        if np.linalg.norm(x_new - x, ord=np.inf) < prec:
            return x_new, k + 1
        x = x_new

    # Non-convergence
    raise ValueError(f"La méthode n'a pas convergé après {nitermax} itérations.")

---

#### II.1.3 - Test de la  fonction de relaxation
> Tester la méthode de relaxation pour la résolution du système linéaire $Ax = b$ pour $n=20$ pour différentes valeurs de $\omega$

In [ ]:
n = 20
A = mat_tridiag(n)
b = vect(n)
prec = 1e-6
nitermax = 1000
omega_values = [0.8, 1.0, 1.2, 1.5, 1.8]

print("Test de la méthode de relaxation (SOR) avec n =", n)

for omega in omega_values:
    try:
        x, niter = relaxation(A, b, prec, nitermax, omega)
        print(f"ω = {omega:.1f} : Convergence atteinte en {niter} itérations")
    except ValueError as e:
        print(f"ω = {omega:.1f} : {e}")

Test de la méthode de relaxation (SOR) avec n = 20
ω = 0.8 : Convergence atteinte en 661 itérations
ω = 1.0 : Convergence atteinte en 458 itérations
ω = 1.2 : Convergence atteinte en 316 itérations
ω = 1.5 : Convergence atteinte en 164 itérations
ω = 1.8 : Convergence atteinte en 64 itérations


La méthode de relaxation necessite donc moins d'itération lorsque qu'elle est sur-relaxé. Ce qui semble logique. Et pour $\omega = 1$ on affirme que la méthode de Gauss-Seidel prend 458 itération à s'exécuté.

---
#### II.1.4 - Convergence de la méthode de relaxation
> Vérifier (expérimentalement) que si la méthode de la relaxation converge pour toute donnée initiale, alors $0<\omega<2$

In [ ]:
# Tester la convergence pour différentes valeurs de ω
n = 20  # Taille de la matrice
A = np.diag(2 * np.ones(n)) + np.diag(-1 * np.ones(n-1), 1) + np.diag(-1 * np.ones(n-1), -1)
b = np.zeros(n)
b[0] = 1
b[-1] = 1

# Paramètres de test
prec = 1e-6
nitermax = 1000
omega_values = [0.1, 0.5, 1.0, 1.5, 1.8, 1.9, 2.0]

# Test de convergence pour chaque ω
results = {}
for omega in omega_values:
    try:
        x, niter = relaxation(A, b, prec, nitermax, omega)
        results[omega] = (x, niter)
    except ValueError as e:
        results[omega] = (None, str(e))

# Affichage des résultats
for omega, (result, niter) in results.items():
    if result is not None:
        print(f"ω = {omega}: Convergence atteinte en {niter} itérations")
    else:
        print(f"ω = {omega}: {niter}")


ω = 0.1: La méthode n'a pas convergé après 1000 itérations.
ω = 0.5: La méthode n'a pas convergé après 1000 itérations.
ω = 1.0: Convergence atteinte en 458 itérations
ω = 1.5: Convergence atteinte en 164 itérations
ω = 1.8: Convergence atteinte en 64 itérations
ω = 1.9: Convergence atteinte en 133 itérations
ω = 2.0: La méthode n'a pas convergé après 1000 itérations.


---

### II.2 - Méthode du gradient à pas fixe

> On rappelle (ou pas) que la méthode du gradient à pas fixe pour une matrice symétrique définie positive, est donnée par l'algorithme suivant : $X^{(0)}$ donné $\in \mathbb{R}^{n}$
>
>$$M X^{(k+1)} = NX^{(k)} + b , k\leqslant 0 $$
>
>avec $M = \frac 1 \alpha$ et $N = \frac 1 \alpha - A$ où $\alpha \in \mathbb{R}^*$ et $I$ est la matrice identité de taille $n$.

---

#### II.2.1 - Fonction gradient à pas fixe
>Créer une fonction `gradientpasfixe(A, b, prec, nitermax, alpha)` qui retourne un couple `(x, niter)`, où :  
>- $x$ est la solution approximative du système $Ax = b$, calculée grâce à la méthode du gradient à pas fixe avec paramètre $\alpha$.
>- `niter` est le nombre d’itérations nécessaires pour atteindre la précision `prec`.
>- La donnée `nitermax` est le nombre maximal d’itérations autorisé, au-delà duquel le programme s’arrête avec un message indiquant la non-convergence

In [ ]:
def gradientpasfixe(A,b,prec,nitermax,alpha):
    n = len(b)
    x = np.zeros(n)
    niter = 0

    I = np.eye(n)
    M = I / alpha
    N = M - A

    while niter < nitermax and max(abs(A @ x - b)) > prec:
        x = M @ (N @ x + b)
        niter += 1

    if niter >= nitermax:
        print("Pas de convergence")

    return x, niter

---

#### II.2.1 - Test et illustration
>On admet que l’algorithme du gradient à pas fixe converge si et seulement si :  
>$$
>\alpha \in ]0, \frac{2}{\lambda_n}[,
>$$  
>où $\lambda_n$ est la plus grande valeur propre de $A$.  
>
>Et que la convergence est optimale si :  
>$$
>\alpha = \frac{2}{\lambda_1 + \lambda_n},
>$$  
>où $\lambda_1$ est la plus petite valeur propre de $A$.
>
>Tester la méthode du gradient à pas fixe pour la résolution du système linéaire $Ax = b$ défini au $1$ dans le cas $n = 10$ pour différentes valeurs de $\alpha$, et illustrer les résultats théoriques précédemment annoncés.

Les valeurs propres sont ici de la forme

$$\lambda_k = 2 - 2 \cos \left( \frac{k\pi}{n+1}\right), \quad k = 1, \dots, n$$

Pour $n = 10$, on trouve :
$$\lambda_{\text{min}} \approx 0,0810 \quad \text{et} \quad \lambda_{\text{max}} \approx 3,918$$

Donc :
$$\frac{2}{\lambda_{\text{max}}} \approx 0,51 \quad \text{et} \quad \alpha_{opt} = \frac{2}{\lambda_{\text{max}} + \lambda_{\text{min}}} \approx 0,5$$

In [ ]:
alpha_opt = 0.5
alpha = 0.2

A = mat_tridiag(10)
b = vect(10)
for i in range(1, 32):
    alpha += 0.01
    x, niter = gradientpasfixe(A, b, 1e-8, 10000, alpha)
    print(f"Pour alpha={alpha} : solution trouvée en {niter} itérations.")

Pour alpha=0.21000000000000002 : solution trouvée en 230 itérations.
Pour alpha=0.22000000000000003 : solution trouvée en 238 itérations.
Pour alpha=0.23000000000000004 : solution trouvée en 245 itérations.
Pour alpha=0.24000000000000005 : solution trouvée en 252 itérations.
Pour alpha=0.25000000000000006 : solution trouvée en 260 itérations.
Pour alpha=0.26000000000000006 : solution trouvée en 267 itérations.
Pour alpha=0.2700000000000001 : solution trouvée en 275 itérations.
Pour alpha=0.2800000000000001 : solution trouvée en 283 itérations.
Pour alpha=0.2900000000000001 : solution trouvée en 291 itérations.
Pour alpha=0.3000000000000001 : solution trouvée en 300 itérations.
Pour alpha=0.3100000000000001 : solution trouvée en 308 itérations.
Pour alpha=0.3200000000000001 : solution trouvée en 317 itérations.
Pour alpha=0.3300000000000001 : solution trouvée en 326 itérations.
Pour alpha=0.34000000000000014 : solution trouvée en 335 itérations.
Pour alpha=0.35000000000000014 : solution

/tmp/ipykernel_1871/2804131006.py:11: RuntimeWarning: overflow encountered in matmul
  x = M @ (N @ x + b)
/tmp/ipykernel_1871/2804131006.py:10: RuntimeWarning: overflow encountered in matmul
  while niter < nitermax and max(abs(A @ x - b)) > prec:
/tmp/ipykernel_1871/2804131006.py:10: RuntimeWarning: invalid value encountered in matmul
  while niter < nitermax and max(abs(A @ x - b)) > prec:
/tmp/ipykernel_1871/2804131006.py:11: RuntimeWarning: invalid value encountered in matmul
  x = M @ (N @ x + b)


---